# EduAssess — ML Module
## Student Performance Prediction using Decision Tree

**Algorithm:** Decision Tree Classifier  
**Task:** Predict student performance category — Good Performer / Average Performer / Needs Improvement  
**Input Features:** Exam Score, Time Taken, Number of Attempts, Past Score  

## Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print('All imports successful!')

## Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('dataset.csv')
print(f'Shape: {df.shape}')
print(f'\nClass Distribution:')
print(df['performance'].value_counts())
df.head(10)

## Step 3 — Data Preprocessing & Encoding

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# Feature and target
FEATURES = ['score', 'time_taken', 'num_attempts', 'past_score']
X = df[FEATURES]
y = df['performance']

# Label Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('\nLabel Encoding:')
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f'  {idx} → {cls}')

print('\nFeature Statistics:')
X.describe()

## Step 4 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## Step 5 — Train Decision Tree Model

In [ ]:
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    criterion='gini'
)
model.fit(X_train, y_train)

print(f'Model trained!')
print(f'Tree Depth  : {model.get_depth()}')
print(f'Tree Leaves : {model.get_n_leaves()}')

print('\nFeature Importances:')
for feat, imp in zip(FEATURES, model.feature_importances_):
    print(f'  {feat:<15} {imp:.4f}')

## Step 6 — Evaluation

In [ ]:
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
cv     = cross_val_score(model, X, y_encoded, cv=5)

print(f'Test Accuracy : {acc*100:.2f}%')
print(f'CV  Accuracy  : {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Step 7 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# Feature Importance
pd.Series(model.feature_importances_, index=FEATURES).sort_values().plot(
    kind='barh', ax=axes[1], color='#4f8ef7'
)
axes[1].set_title('Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree Structure
plt.figure(figsize=(18, 8))
plot_tree(model, feature_names=FEATURES, class_names=le.classes_,
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Structure', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8 — Save Model (pickle)

In [ ]:
model_data = {
    'model':         model,
    'label_encoder': le,
    'features':      FEATURES,
    'accuracy':      acc,
    'classes':       list(le.classes_)
}

# Save in ML/ folder
with open('model.pkl', 'wb') as f:
    pickle.dump(model_data, f)
print('Saved: ML/model.pkl')

# Also copy to Backend/ml_model/
import os, shutil
backend_path = os.path.join('..', 'Backend', 'ml_model', 'model.pkl')
os.makedirs(os.path.dirname(backend_path), exist_ok=True)
shutil.copy('model.pkl', backend_path)
print('Saved: Backend/ml_model/model.pkl')
print(f'\nFinal Accuracy: {acc*100:.2f}%')

## Step 9 — Test Prediction

In [ ]:
# Load saved model and test
with open('model.pkl', 'rb') as f:
    saved = pickle.load(f)

clf = saved['model']
le2 = saved['label_encoder']

test_cases = [
    [88, 30, 2, 85],   # Expected: Good
    [62, 38, 3, 60],   # Expected: Average
    [35, 42, 5, 40],   # Expected: Needs Improvement
]

for tc in test_cases:
    X_new = pd.DataFrame([tc], columns=FEATURES)
    pred  = le2.inverse_transform(clf.predict(X_new))[0]
    conf  = round(max(clf.predict_proba(X_new)[0]) * 100, 1)
    print(f'Score={tc[0]}% → {pred} ({conf}% confidence)')